# Partial trust: the readings' motion

One run of utter's `scripts/partial_trust.py` over the Speech Commands testing and validation splits, exported by `export/trust.py`. Two tables: `clips`, one row per clip with what the harness recorded at the first sighting of the first word and whether that word survived to the final; `advances`, one row per clip, decoder advance and reading from the sighting on, with each reading's lead over the best other reading, its velocity (`lead_delta`, the lead's change since the previous advance) and its relation to rank 0.

Every figure is Plotly: drag to zoom, double-click to reset, box-select, hover for the clip. The manifest prints first, because a figure is only as good as the runtime that decoded its audio.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

RUN = Path("../data/2026-09-12-trust")
manifest = json.loads((RUN / "MANIFEST.json").read_text())
clips = pd.read_parquet(RUN / "clips.parquet")
adv = pd.read_parquet(RUN / "advances.parquet")
test = clips[clips.split == "testing"]

# categorical slots 1 and 2, and three ordinal steps of one hue, from the validated palette
COL = {"survived": "#2a78d6", "revised": "#eb6834"}
BAND = {"<1": "#86b6ef", "1-4": "#3987e5", ">=4": "#1c5cab"}
def outcome(s):
    return np.where(s, "survived", "revised")

print(json.dumps({k: manifest[k] for k in ("exported", "harness", "harness_args", "wheel", "utter_head", "utter_dirty")}, indent=1))
clips.groupby("split").survived.agg(clips="count", revised=lambda s: int((~s).sum()))

## Calibration: does the gap's sigmoid predict survival?

The gap is rank 0's lead over rank 1 at the sighting. Bars are the observed survival rate in each gap bucket on the testing split; the marker is `sigmoid(mean gap)` for that bucket, the prediction the README gives a host.

In [ ]:
edges = [0, 0.5, 1, 2, 4, 8, np.inf]
labels = ["0-0.5", "0.5-1", "1-2", "2-4", "4-8", "8+"]
t = test.assign(bucket=pd.cut(test.gap, edges, labels=labels, right=False))
cal = t.groupby("bucket", observed=True).agg(n=("gap", "size"), survived=("survived", "mean"), mean_gap=("gap", "mean"))
cal["predicted"] = 1 / (1 + np.exp(-cal.mean_gap))
fig = go.Figure()
fig.add_bar(x=cal.index.astype(str), y=cal.survived, name="observed survival", marker_color=COL["survived"],
            text=cal.n, textposition="outside", hovertemplate="%{x} nats: %{y:.0%} of %{text} clips<extra></extra>")
fig.add_scatter(x=cal.index.astype(str), y=cal.predicted, name="sigmoid(mean gap)", mode="markers",
                marker=dict(color="#0b0b0b", size=11, symbol="diamond"))
fig.update_layout(template="plotly_white", yaxis=dict(title="first word survived", tickformat=".0%", range=[0, 1.08]),
                  xaxis_title="gap at the sighting (nats)", height=420, legend=dict(orientation="h"))
fig

## The first-shown reading, advance by advance

Each thin line is one clip: the lead of the reading that was rank 0 at the sighting, at the sighting and at each advance after it (advances are 240 ms apart on this model). Blue held to the final, orange was revised. Bold lines are the medians. A revised reading's lead is the one that crosses zero: the next chunk's evidence reversed it.

In [ ]:
t0 = adv[(adv.adv == 0) & (adv["rank"] == 0)][["clip_id", "text"]].rename(columns={"text": "t0"})
first = adv.merge(t0, on="clip_id").merge(test[["clip_id", "survived"]], on="clip_id")
first = first[first.text == first.t0].assign(ms_after=lambda d: d.adv * 240)

def nan_lines(df, x, y):
    xs, ys = [], []
    for _, g in df.groupby("clip_id", sort=False):
        xs += list(g[x]) + [None]
        ys += list(g[y]) + [None]
    return xs, ys

fig = go.Figure()
for name, sel in (("survived", first.survived), ("revised", ~first.survived)):
    g = first[sel].sort_values(["clip_id", "adv"])
    xs, ys = nan_lines(g, "ms_after", "lead")
    fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", name=f"{name} ({g.clip_id.nunique()} clips)",
                               line=dict(color=COL[name], width=1), opacity=0.08, hoverinfo="skip"))
    med = g.groupby("adv").lead.median()
    fig.add_trace(go.Scatter(x=med.index * 240, y=med.values, mode="lines+markers", name=f"{name} median",
                             line=dict(color=COL[name], width=3), marker=dict(size=9)))
fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
fig.update_layout(template="plotly_white", xaxis=dict(title="ms after the sighting", tickvals=[0, 240, 480]),
                  yaxis_title="lead over the best other reading (nats)", height=520)
fig

## Momentum: this advance's velocity against the next

For every reading present at three consecutive advances, its velocity at advance k against its velocity at k+1, coloured by how far ahead or behind it stood at k. Persistence lives in the 1-4 nat band; past 4 a won contest stops moving. The dotted line is equality.

In [ ]:
a = adv[adv.split == "testing"].sort_values(["clip_id", "text", "adv"]).copy()
grp = a.groupby(["clip_id", "text"])
a["next_delta"] = grp.lead_delta.shift(-1)
a["next_adv"] = grp.adv.shift(-1)
pairs = a[(a.next_adv == a.adv + 1) & a.lead_delta.notna() & a.next_delta.notna()].copy()
pairs["band"] = pd.cut(pairs.lead.abs(), [0, 1, 4, np.inf], labels=list(BAND), right=False)
for b, g in pairs.groupby("band", observed=True):
    same = ((g.lead_delta > 0) == (g.next_delta > 0)).mean()
    print(f"|lead| {b:>4}: n={len(g):5d}  r={g.lead_delta.corr(g.next_delta):+.3f}  same sign {same:.0%}")

fig = go.Figure()
for b in BAND:
    g = pairs[pairs.band == b]
    fig.add_trace(go.Scattergl(x=g.lead_delta, y=g.next_delta, mode="markers", name=f"|lead| {b} (n={len(g)})",
                               marker=dict(color=BAND[b], size=6, opacity=0.55),
                               text=g.clip_id + "  " + g["text"], hovertemplate="%{text}<br>now %{x:.2f}<br>next %{y:.2f}<extra></extra>"))
lim = float(np.nanmax(np.abs(pairs[["lead_delta", "next_delta"]].values)))
fig.add_shape(type="line", x0=-lim, y0=-lim, x1=lim, y1=lim, line=dict(color="#c3c2b7", dash="dot"))
fig.update_layout(template="plotly_white", xaxis_title="lead_delta at advance k (nats)",
                  yaxis_title="lead_delta at advance k+1 (nats)", height=560)
fig

## At the sighting: standing against new evidence

Left: the gap (where the leader stands) against the displaced reading's velocity (how hard the previous leader fell), where that is defined. Right: the gap against the entropy of the readings' softmax, the one always-available figure that separates outcomes at equal gap.

In [ ]:
d = test.assign(outcome=outcome(test.survived))
f1 = px.scatter(d[d.displaced_delta.notna()], x="gap", y="displaced_delta", color="outcome", color_discrete_map=COL,
                hover_data=["clip_id", "label", "top_word"], render_mode="webgl", opacity=0.55, template="plotly_white",
                labels={"gap": "gap at the sighting (nats)", "displaced_delta": "displaced reading's lead_delta (nats)"}, height=480)
f1.show()
f2 = px.scatter(d, x="gap", y="entropy", color="outcome", color_discrete_map=COL, hover_data=["clip_id", "label", "top_word"],
                render_mode="webgl", opacity=0.45, template="plotly_white",
                labels={"gap": "gap at the sighting (nats)", "entropy": "entropy of the readings' softmax (nats)"}, height=480)
f2

## One clip, every reading

Every reading's lead across the clip's advances, like one batch's tags on a trend. Hairlines are the decoder's advances, the dashed line the sighting. Call `trend(clip_id)` with any id from the table below.

In [ ]:
def trend(clip):
    c = clips[clips.clip_id == clip].iloc[0]
    g = adv[adv.clip_id == clip]
    fig = go.Figure()
    for text, s in g.groupby("text", sort=False):
        s = s.sort_values("adv")
        fig.add_trace(go.Scatter(x=s.ms, y=s.lead, mode="lines+markers+text", name=text,
                                 text=[""] * (len(s) - 1) + [text], textposition="middle right",
                                 customdata=s.lead_delta.round(2),
                                 hovertemplate=f"{text}<br>%{{x}} ms<br>lead %{{y:.2f}}<br>velocity %{{customdata}}<extra></extra>"))
    for ms in json.loads(c.advance_ms):
        fig.add_vline(x=ms, line=dict(color="#e1e0d9", width=1))
    fig.add_vline(x=c.sighting_ms, line=dict(color="#898781", dash="dash"))
    fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
    fig.update_layout(template="plotly_white", height=480, xaxis_title="ms of audio fed",
                      yaxis_title="lead over the best other reading (nats)",
                      title=f"{clip}: said {c.label}, first shown {c.top_word}, {'survived' if c.survived else 'revised'}")
    return fig

revised_with_a_big_lead = test[~test.survived].sort_values("gap", ascending=False).iloc[0].clip_id
trend(revised_with_a_big_lead)

In [ ]:
trend(test[test.survived & (test.n_advances == 3)].sample(1, random_state=7).iloc[0].clip_id)

## Table

The testing clips, revisions first, as a place to pick ids from. `pd.set_option("display.max_rows", ...)` or filter as you like.

In [ ]:
cols = ["clip_id", "label", "top_word", "survived", "gap", "entropy", "displaced_delta", "lead_delta0", "had_history", "sighting_ms", "n_advances"]
test.sort_values(["survived", "gap"], ascending=[True, False])[cols].head(40)

## The page's own figures

The tables are a re-derivation, so they have to agree with the page the harness wrote from the same run. This cell fails if they do not.

In [ ]:
fig = json.loads((RUN / "raw" / "partial-trust.json").read_text())
assert len(test) == fig["first_sightings"], (len(test), fig["first_sightings"])
assert int((~test.survived).sum()) == fig["revised"]
# the page's buckets carry their lower edge only; the upper edge is the next row's
bounds = [r["lo"] for r in fig["calibration"]] + [np.inf]
for row, hi in zip(fig["calibration"], bounds[1:]):
    b = test[(test.gap >= row["lo"]) & (test.gap < hi)]
    assert len(b) == row["n"] and int(b.survived.sum()) == row["survived"], row
assert int(test.had_history.sum()) == fig["availability"]["defined"]["history"]
assert int(test.lead_delta0.notna().sum()) == fig["availability"]["defined"]["lead_delta0"]
print("clips, revisions, every gap bucket and the availability counts match", fig["run"])